# Census MTIS 2018–2022 制造业合计库存销售比原始来源

完整Stage1 f696+b472为父方案；本卡只接收官方目录锁定的60个月原始PDF，保留初值、同版上期修订、发布时间、错误与NAS备份。0 SOXX结果读取、0拟合、0账户。

In [1]:
import datetime,email.utils,hashlib,io,json,os,pathlib,re,shlex,subprocess,sys,tarfile,tempfile,time,urllib.parse
from bs4 import BeautifulSoup
from pypdf import PdfReader
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
root=pathlib.Path.cwd() if (pathlib.Path.cwd()/'src').exists() else pathlib.Path.cwd().parents[1];sys.path.insert(0,str(root))
from src.data.nas_market_data import nas_api_call,archive_nas_recorder,_settings,_ssh
card=root/'research/stage1_census_mtis_2018_2022_original_source_20261023';snap=root/'data/snapshots/stage1_census_mtis_2018_2022_original_source_20261023';snap.mkdir(parents=True,exist_ok=True);plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
index=root/src['official_index_saved_path'];sample=root/src['sample_card'];sample_summary=json.loads((sample/'summary.json').read_text());assert sha(index)==src['official_index_sha256'] and sample_summary['source_index_NAS_sha256']==src['official_index_NAS_sha256'] and sample_summary['source_PDF_NAS_sha256']==src['reused_2018Jul_NAS_group_sha256'] and sample_summary['sources_cold_restored'] and sha(sample/'summary.json')==src['sample_summary_sha256'];reused=root/src['reused_2018Jul_original_path'];assert sha(reused)==src['reused_2018Jul_original_sha256']
soup=BeautifulSoup(index.read_bytes(),'html.parser');links={urllib.parse.urljoin('https://www.census.gov/mtis/historic_releases.html',a.get('href')) for a in soup.find_all('a') if a.get('href')};months=[f'{y}-{m:02d}' for y in range(2018,2023) for m in range(1,13)];urls={m:f'https://www2.census.gov/mtis/historical/mtis{int(m[:4])%100:02d}{int(m[5:]):02d}.pdf' for m in months};assert len(months)==len(set(urls.values()))==60 and all(u in links for u in urls.values())
state={'status':'capturing_original_PDFs','plan_sha256':sha(card/'plan.json'),'GETs_started':0,'saved_PDF_reads':1,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');receipts=[]
for m in months:
 u=urls[m]
 if m=='2018-07':receipts.append({'month':m,'url':u,'source':'reused_exact_NAS_restored_PDF','status':'reused','http_status':200,'body_bytes':reused.stat().st_size,'body_sha256':sha(reused),'local_path':str(reused.relative_to(root)),'headers_path':None,'error':None});continue
 state['GETs_started']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');body=snap/(m+'.response');headers=snap/(m+'.headers');start=datetime.datetime.now(datetime.timezone.utc).isoformat();proc=subprocess.run(['curl','--http1.1','--location','--silent','--show-error','--connect-timeout','5','--max-time','20','--user-agent','soxl-qlib-source-research','--dump-header',str(headers),'--output',str(body),'--write-out','%{http_code}|%{url_effective}',u],capture_output=True,text=True);parts=proc.stdout.strip().split('|',1)
 r={'month':m,'url':u,'source':'new_exact_GET','started_utc':start,'ended_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'curl_exit':proc.returncode,'http_status':int(parts[0]) if parts and parts[0].isdigit() else None,'final_url':parts[1] if len(parts)>1 else None,'error':proc.stderr[:200] or None,'body_bytes':body.stat().st_size if body.exists() else 0,'body_sha256':sha(body) if body.exists() else None,'local_path':str(body.relative_to(root)) if body.exists() else None,'headers_path':str(headers.relative_to(root)) if headers.exists() else None,'headers_sha256':sha(headers) if headers.exists() else None,'within_size_cap':body.exists() and body.stat().st_size<=2000000};receipts.append(r);time.sleep(.05)
(snap/'response_receipts.json').write_text(json.dumps(receipts,ensure_ascii=False,indent=2)+'\n')
qlib.init(provider_uri=str(root/'data/qlib/soxl'),region='us',kernels=1);calendar=[str(x)[:10] for x in D.calendar(start_time='2018-01-01',end_time='2023-04-30',freq='day')];mon=['JANUARY','FEBRUARY','MARCH','APRIL','MAY','JUNE','JULY','AUGUST','SEPTEMBER','OCTOBER','NOVEMBER','DECEMBER'];rows=[];errors=[]
for r in receipts:
 m=r['month'];c={'economic_month':m,'source_sha256':r['body_sha256'],'source_bytes':r['body_bytes'],'source_kind':r['source'],'HTTP_status':r['http_status'],'release_date':None,'release_time_ET':None,'first_Qlib_session_strictly_after_release':None,'manufacturers_adjusted_current_preliminary_ratio':None,'manufacturers_adjusted_previous_revised_ratio':None,'current_byte_first_public_qualified':False,'server_Last_Modified_UTC':None,'server_Last_Modified_after_release_date':None,'status':'unknown'}
 if r['source']=='reused_exact_NAS_restored_PDF' or (r.get('curl_exit')==0 and r['http_status']==200 and r['within_size_cap']):
  path=root/r['local_path'];data=path.read_bytes()
  if data.startswith(b'%PDF'):
   try:
    reader=PdfReader(io.BytesIO(data));assert 1<=len(reader.pages)<=12;t='\n'.join(pg.extract_text() or '' for pg in reader.pages);head=t[:1800].upper();date=re.search(r'(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)\s+(\d{1,2}),\s+(20\d{2})',head);time_ok=bool(re.search(r'10:00\s*(?:A\.?M\.?)?\s*E(?:D|S)T',head));economic_title=mon[int(m[5:])-1]+' '+m[:4] in head
    if date and time_ok and economic_title:
     release=f'{int(date.group(3)):04d}-{mon.index(date.group(1))+1:02d}-{int(date.group(2)):02d}';c['release_date']=release;c['release_time_ET']='10:00';c['first_Qlib_session_strictly_after_release']=next((d for d in calendar if d>release),None)
    start=re.search(r'(?im)^\s*Adjusted\s*\d*\s*$',t);adjusted=t[start.end():].split('Not Adjusted',1)[0] if start else '';matches=list(re.finditer(r'Manufacturers(?:²|3|2)?[^\d]{0,100}?((?:\d[\d,]*\s+){6})(\d+\.\d\d)\s+(\d+\.\d\d)\s+(\d+\.\d\d)',adjusted,re.I|re.S))
    if len(matches)==1:
     z=matches[0];c['manufacturers_adjusted_current_preliminary_ratio']=float(z.group(2));c['manufacturers_adjusted_previous_revised_ratio']=float(z.group(3));c['same_release_prior_year_ratio']=float(z.group(4))
    else:c['parse_diagnostic']={'adjusted_Manufacturers_matches':len(matches),'adjusted_chars':len(adjusted),'release_date_found':bool(date),'clock_found':time_ok,'economic_title_found':economic_title}
    if c['release_date'] and c['first_Qlib_session_strictly_after_release'] and c['manufacturers_adjusted_current_preliminary_ratio'] is not None:c['status']='current_original_source_semantics_and_clock_qualified'
   except Exception as e:c['parse_error']=type(e).__name__+': '+str(e)[:180]
  else:c['parse_error']='PDF_magic_missing'
 if r.get('headers_path'):
  raw=(root/r['headers_path']).read_text(errors='replace');lm=re.findall(r'(?im)^Last-Modified:\s*(.+?)\s*$',raw)
  if lm:
   c['server_Last_Modified_UTC']=lm[-1]
   if c['release_date']:
    try:c['server_Last_Modified_after_release_date']=email.utils.parsedate_to_datetime(lm[-1]).date().isoformat()>c['release_date']
    except Exception:pass
 rows.append(c)
 if c['status']=='unknown':errors.append({'month':m,'http_status':r['http_status'],'transport_error':r['error'],'parse_error':c.get('parse_error'),'diagnostic':c.get('parse_diagnostic')})
table=pd.DataFrame(rows);assert len(table)==60 and table.economic_month.tolist()==months;complete=not errors;result={'status':'current_original_MTiS_2018_2022_60_month_source_complete_PIT_pending' if complete else 'current_original_MTiS_2018_2022_source_months_unknown','plan_sha256':sha(card/'plan.json'),'source_months':60,'external_GETs_started':state['GETs_started'],'saved_PDF_reads':1,'HTTP200_or_reused':sum(r['http_status']==200 for r in receipts),'source_semantics_clock_qualified_months':int(table.status.eq('current_original_source_semantics_and_clock_qualified').sum()),'unknown_months':[x['month'] for x in errors],'unknown_reasons':errors,'current_bytes_first_public_qualified':False,'server_Last_Modified_after_release_date_count':int(table.server_Last_Modified_after_release_date.eq(True).sum()),'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
settings=_settings(root);files=sorted(x for x in snap.iterdir() if x.is_file());entries={str(x.relative_to(root)):sha(x) for x in files}
with tempfile.TemporaryDirectory(prefix='mtis-train-source-') as tmp:
 archive=pathlib.Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for x in files:t.add(x,arcname=str(x.relative_to(root)),recursive=False)
 digest=sha(archive);nas=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert nas['sha256']==digest and nas['size_bytes']==archive.stat().st_size;remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='mtis-train-source-restore-') as back:
  copy=pathlib.Path(back)/'source.tar.gz'
  with copy.open('wb') as out:subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,check=True,timeout=150)
  assert sha(copy)==digest
  with tarfile.open(copy,'r:gz') as t:
   members={m.name:m for m in t.getmembers() if m.isfile()};assert set(members)==set(entries)
   for rel,h in entries.items():assert hashlib.sha256(t.extractfile(members[rel]).read()).hexdigest()==h
source_archive={**nas,'nas_path':remote,'files':len(files),'all_member_cold_restore_verified':True,'reused_2018Jul_prior_NAS_sha256':src['reused_2018Jul_NAS_group_sha256']};(card/'source_archive.json').write_text(json.dumps(source_archive,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_archive=source_archive);rec.save_objects(artifact_path='analysis',monthly_source=table,unknown_reasons=errors,result=result);rec.log_metrics(source_months=60,qualified_months=result['source_semantics_clock_qualified_months'],external_GETs=state['GETs_started'],model_fits=0,native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state.update(status='finished',recorder_id=rid,experiment_id=eid);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');ev=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(ev['nas_path'])).split()[0]==ev['sha256']
summary={'experiment':plan['experiment'],'decision':result['status'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_archive':'source_archive.json','source_NAS_sha256':digest,'source_NAS_cold_restore_verified':True,'reused_2018Jul_NAS_sha256':src['reused_2018Jul_NAS_group_sha256'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':ev,'NAS_readback':{'sha256':ev['sha256'],'remote_hash_verified':True},'qualified_months':result['source_semantics_clock_qualified_months'],'unknown_months':result['unknown_months'],'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/monthly_source','analysis/unknown_reasons','analysis/result']},'limitations':['Current original-PDF bytes and mutable Last-Modified metadata do not certify historical first-public vintages.','Manufacturers aggregate is not semiconductor-specific or real-price adjusted.','No Stage1 information fit, complete 0bp funded account or SOXX strategy success.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');print(json.dumps({'decision':result['status'],'qualified':result['source_semantics_clock_qualified_months'],'unknown':result['unknown_months'],'late_last_modified':result['server_Last_Modified_after_release_date_count'],'recorder_id':rid,'NAS_sha256':digest},ensure_ascii=False))


2026/10/23 14:58:06 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[12857:MainThread](2026-10-23 15:02:40,889) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[12857:MainThread](2026-10-23 15:02:40,894) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[12857:MainThread](2026-10-23 15:02:40,895) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[12857:MainThread](2026-10-23 15:02:51,605) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_census_mtis_2018_2022_original_source_20261023.


[12857:MainThread](2026-10-23 15:02:51,608) INFO - qlib.workflow - [exp.py:258] - Experiment 241675816966146999 starts running ...


[12857:MainThread](2026-10-23 15:02:51,681) INFO - qlib.workflow - [recorder.py:345] - Recorder 24ad64dd9d1f4c96b281bbd9dd0fe72d starts running under Experiment 241675816966146999 ...


[12857:MainThread](2026-10-23 15:02:52,433) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{"decision": "current_original_MTiS_2018_2022_source_months_unknown", "qualified": 41, "unknown": ["2020-06", "2020-07", "2020-12", "2021-01", "2021-02", "2021-03", "2021-04", "2021-11", "2021-12", "2022-01", "2022-02", "2022-03", "2022-04", "2022-05", "2022-06", "2022-07", "2022-08", "2022-09", "2022-10"], "late_last_modified": 0, "recorder_id": "24ad64dd9d1f4c96b281bbd9dd0fe72d", "NAS_sha256": "3be204a3695cf40ca90ea18182fb273a272baacc7931b9e1dc33066a6cd4616e"}


# Census MTIS 2018–2022 原始月报第一版结论

冻结60个月来源清单、59次新GET与一份先前NAS恢复的2018年7月PDF。60个月均有官方HTTP200或精确复用原件；全部原始响应、头与失败状态均NAS逐件冷恢复。固定解析接受 **41/60** 月，其余16个月因“Not Adjusted”大小写使调整/未调整行同时匹配、3个月PDF下载到部分字节后超时而保留未知。该固定版本没有重试或填值。

另冻的[机械修复](../stage1_census_mtis_2018_2022_source_repair_20261023/conclusion.md)只改变分隔符大小写识别，并对三个精确超时URL各重取一次；原先41行逐字段不变。此卡0 SOXX收益读取、0拟合、0账户，不能据其部分来源面运行策略。
